In [ ]:
# Add this at the VERY TOP of Cell 2, before any imports
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
# 1: Set Up & Utilities
import os, time, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")
 
# 🌱 Set seed globally
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
 
# 🌱 Multi-seed wrapper
def run_with_seeds(train_fn, seeds=[42, 123, 7], **kwargs):
    metrics = {"acc": [], "f1": [], "train_time": [], "inf_time": []}
    raw_results = []
 
    for s in seeds:
        set_seed(s)
        res = train_fn(seed=s, **kwargs)
        raw_results.append({"seed": s, **res})
 
        for k in metrics:
            if f"test_{k}" in res:
                metrics[k].append(res[f"test_{k}"])
            elif k in res:
                metrics[k].append(res[k])
            else:
                raise KeyError(f"Missing metric '{k}' in result for seed {s}")
 
    summary = {
        k: {
            "mean": float(np.mean(v)),
            "std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
        }
        for k, v in metrics.items()
    }
 
    return {"summary": summary, "raw_results": raw_results}
 
# 📏 Text Length & Truncation Analysis
def analyze_lengths(texts, tokenizer, max_len=512):
    lens = [len(tokenizer.encode(t, truncation=False)) for t in texts]
    stats = {
        "mean": float(np.mean(lens)),
        "median": float(np.median(lens)),
        "p95": float(np.percentile(lens, 95)),
        "max": int(np.max(lens)),
        "truncated": int(sum(l > max_len for l in lens)),
        "total": int(len(lens)),
        "truncated_pct": float(100 * sum(l > max_len for l in lens) / len(lens))
    }
 
    print(
        f"📊 Length Stats (tokens) | Mean: {stats['mean']:.1f} | "
        f"Median: {stats['median']:.1f} | P95: {stats['p95']:.1f} | Max: {stats['max']}"
    )
    print(
        f"🔪 {stats['truncated']}/{stats['total']} "
        f"({stats['truncated_pct']:.1f}%) exceed {max_len} tokens and will be truncated."
    )
    return stats
 
# ⚖️ Class Imbalance Check
def check_distribution(y, names):
    from collections import Counter
    cnt = Counter(y)
    total = len(y)
    rows = []
    for k, v in sorted(cnt.items()):
        rows.append({
            "class_id": k,
            "class_name": names[k],
            "count": v,
            "pct": 100 * v / total
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


In [ ]:
# 2: Load & Split Datasets

# 2A:  Load & Split Newsgroups
import os
import re
import kagglehub
from sklearn.model_selection import train_test_split

print("📥 Loading 20 Newsgroups from Kaggle...")
base_path = kagglehub.dataset_download("crawford/20-newsgroups")
txt_files = sorted([f for f in os.listdir(base_path) if f.endswith('.txt')])

if not txt_files:
    raise FileNotFoundError("No .txt files found in Kaggle download.")

class_names = [f.replace('.txt', '') for f in txt_files]
label_map = {name: i for i, name in enumerate(class_names)}
print(f"📂 Found {len(class_names)} newsgroup files")

all_texts = []
all_labels = []

for cls in class_names:
    file_path = os.path.join(base_path, f"{cls}.txt")
    if not os.path.exists(file_path):
        continue
        
    with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
        raw = f.read()
        
    if not raw.strip():
        print(f"⚠️ {cls}.txt is empty. Skipping.")
        continue

    # Strategy 1: Split by double newline (handles \n\n and \r\n\r\n)
    docs = re.split(r'\r?\n\r?\n', raw)
    
    # Strategy 2: If Strategy 1 fails (< 5 docs), split by Usenet header markers
    if len(docs) < 5:
        docs = re.split(r'(?=^From:|^Subject:|^Article-I\.D\.:|^Path:)', raw, flags=re.MULTILINE)
        
    # Strategy 3: Fallback to single newline if still too few
    if len(docs) < 5:
        docs = raw.split('\n')

    docs_found = 0
    for doc in docs:
        doc = doc.strip()
        if len(doc) < 50:
            continue
            
        # Light cleaning: remove header lines, keep body
        lines = doc.split('\n')
        cleaned = []
        in_header = True
        for line in lines:
            line_stripped = line.strip()
            if in_header:
                if re.match(r'^(From|Subject|Organization|Date|Reply-To|Lines|Distribution|Keywords|Article-I\.D\.|NNTP-Posting-Host|Path|Message-ID|X-Newsreader|X-Trace|Posted|Followup-To):', line_stripped, re.I):
                    continue
                if line_stripped == '':
                    in_header = False
                    continue
                continue
            # Skip quotes/signatures
            if line_stripped.startswith('>') or line_stripped.startswith('|') or line_stripped.startswith('---') or line_stripped.startswith('___'):
                continue
            if line_stripped:
                cleaned.append(line_stripped)
                
        final_text = ' '.join(cleaned)
        if len(final_text) > 60:
            all_texts.append(final_text)
            all_labels.append(label_map[cls])
            docs_found += 1
            
    print(f"   📄 {cls}: {docs_found} docs loaded")

print(f"\n✅ Total 20NG documents: {len(all_texts)}")

if len(all_texts) == 0:
    # Fallback: print exact file stats so we can fix it in 1 message
    print("❌ Still zero documents. Printing file diagnostics:")
    print(f"   File: {file_path}")
    print(f"   Size: {len(raw)} bytes")
    print(f"   First 300 chars:\n{raw[:300]}")
    raise SystemExit("Reply with the diagnostics above so I can give you a 1-line fix.")

# Stratified Split: 80% Train, 10% Val, 10% Test
print("🔀 Splitting data (80/10/10)...")
X_train, X_temp, y_train, y_temp = train_test_split(
    all_texts, all_labels, test_size=0.2, random_state=42, stratify=all_labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

news_train = (X_train, y_train)
news_val   = (X_val, y_val)
news_test  = (X_test, y_test)
news_names = class_names

print(f"\n📊 20 Newsgroups Ready:")
print(f"   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"   Sample text: '{X_train[0][:100]}...'")
print(f"   Sample label: {news_names[y_train[0]]}")

In [ ]:
# 2B: Load & Split TweeEval (Irony)

import os
import glob
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

print("📥 Loading TweetEval from Kaggle...")

# 1. Download
path = kagglehub.dataset_download("thedevastator/tweeteval-a-multi-task-classification-benchmark")
print(f"📁 Dataset downloaded to: {path}")

# 2. Find the Irony files
# Kaggle datasets can be nested; we search recursively for CSVs containing "irony"
all_csvs = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
irony_files = [f for f in all_csvs if "irony" in f.lower()]

if not irony_files:
    raise ValueError(f"Could not find 'irony' files in {path}. Files found: {os.listdir(path)}")

print(f"📄 Found {len(irony_files)} Irony file(s): {[os.path.basename(f) for f in irony_files]}")

# 3. Load Data
dfs = []
for f in irony_files:
    dfs.append(pd.read_csv(f))

if not dfs:
    raise ValueError("Empty CSVs found.")

# Combine train/test into one DataFrame for our own splitting
df = pd.concat(dfs, ignore_index=True)

# 4. Identify Columns
# Try to find the text and label columns automatically
text_col = next((c for c in df.columns if c.lower() in ['text', 'tweet', 'sentence']), df.columns[0])
label_col = next((c for c in df.columns if c.lower() in ['label', 'class', 'target']), df.columns[1])

print(f" Using column '{text_col}' for text and '{label_col}' for labels.")

# Clean data: Remove NaNs and ensure strings
df = df.dropna(subset=[text_col, label_col])
tweet_texts = df[text_col].astype(str).tolist()
tweet_labels = df[label_col].astype(int).tolist()
tweet_names = ["non_ironic", "ironic"] # Standard TweetEval irony classes

print(f"✅ Loaded {len(tweet_texts)} tweets.")

# 5. Stratified Split: 80% Train, 10% Val, 10% Test
print("🔀 Splitting data (80/10/10)...")
X_train, X_temp, y_train, y_temp = train_test_split(
    tweet_texts, tweet_labels, test_size=0.2, random_state=42, stratify=tweet_labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

tweet_train = (X_train, y_train)
tweet_val   = (X_val, y_val)
tweet_test  = (X_test, y_test)

print(f"\n📊 TweetEval Ready:")
print(f"   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"   Sample text: '{X_train[0][:80]}...'")
print(f"   Sample label: {tweet_names[y_train[0]]} (class {y_train[0]})")

In [ ]:
# 3: Classical Models (TF-IDR + LR / SVM)

import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

def train_classical(X_train, y_train, X_val, y_val, X_test, y_test, 
                    model_type="lr", c_values=[0.1, 1.0, 10.0], seed=42):
    np.random.seed(seed)
    best_model = None
    best_val_f1 = -1.0
    best_C = None
    
    for C in c_values:
        if model_type == "lr":
            clf = LogisticRegression(C=C, max_iter=1000, random_state=seed, n_jobs=-1, class_weight='balanced')
        else:
            clf = LinearSVC(C=C, max_iter=2000, random_state=seed, dual=False, class_weight='balanced')
            
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)),
            ('clf', clf)
        ])
        
        t0 = time.time()
        pipe.fit(X_train, y_train)
        train_time = time.time() - t0
        
        y_val_pred = pipe.predict(X_val)
        val_f1 = f1_score(y_val, y_val_pred, average='macro')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_C = C
            best_model = pipe
            
    t0 = time.time()
    y_test_pred = best_model.predict(X_test)
    inf_time_per_sample = (time.time() - t0) / len(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    # 🟢 ADD: Per-class F1 extraction (Rubric requirement)
    report = classification_report(y_test, y_test_pred, output_dict=True)
    cls_f1 = {k: v['f1-score'] for k, v in report.items() 
              if k not in ['accuracy', 'macro avg', 'weighted avg']}
    best_cls = max(cls_f1, key=cls_f1.get) if cls_f1 else None
    worst_cls = min(cls_f1, key=cls_f1.get) if cls_f1 else None
    
    return {
        'best_C': best_C, 'val_f1': best_val_f1,
        'test_acc': test_acc, 'test_f1': test_f1,
        'train_time': train_time, 'inf_time': inf_time_per_sample,
        'best_class': best_cls, 'best_class_f1': cls_f1.get(best_cls, 0.0),  # 🟢 ADD
        'worst_class': worst_cls, 'worst_class_f1': cls_f1.get(worst_cls, 0.0),  # 🟢 ADD
        'model': best_model
    }

# ================= RUN ON 20 NEWGROUPS =================
print("🚀 Training Classical Models on 20 Newsgroups...")

news_lr = run_with_seeds(
    train_classical,
    seeds=[42, 123, 7],
    X_train=news_train[0], y_train=news_train[1],
    X_val=news_val[0], y_val=news_val[1],
    X_test=news_test[0], y_test=news_test[1],
    model_type="lr"
)

news_svm = run_with_seeds(
    train_classical,
    seeds=[42, 123, 7],
    X_train=news_train[0], y_train=news_train[1],
    X_val=news_val[0], y_val=news_val[1],
    X_test=news_test[0], y_test=news_test[1],
    model_type="svm"
)

print("\n📊 20NG Results:")
print(
    f"   LR : Acc={news_lr['summary']['acc']['mean']:.4f} ± {news_lr['summary']['acc']['std']:.4f} | "
    f"F1={news_lr['summary']['f1']['mean']:.4f} ± {news_lr['summary']['f1']['std']:.4f} | "
    f"Train={news_lr['summary']['train_time']['mean']:.2f}s | "
    f"Inf={news_lr['summary']['inf_time']['mean']*1000:.2f}ms/sample"
)
print(
    f"   SVM: Acc={news_svm['summary']['acc']['mean']:.4f} ± {news_svm['summary']['acc']['std']:.4f} | "
    f"F1={news_svm['summary']['f1']['mean']:.4f} ± {news_svm['summary']['f1']['std']:.4f} | "
    f"Train={news_svm['summary']['train_time']['mean']:.2f}s | "
    f"Inf={news_svm['summary']['inf_time']['mean']*1000:.2f}ms/sample"
)

# ================= RUN ON TWEETEVAL =================
print("\n🚀 Training Classical Models on TweetEval...")

tweet_lr = run_with_seeds(
    train_classical,
    seeds=[42, 123, 7],
    X_train=tweet_train[0], y_train=tweet_train[1],
    X_val=tweet_val[0], y_val=tweet_val[1],
    X_test=tweet_test[0], y_test=tweet_test[1],
    model_type="lr"
)

tweet_svm = run_with_seeds(
    train_classical,
    seeds=[42, 123, 7],
    X_train=tweet_train[0], y_train=tweet_train[1],
    X_val=tweet_val[0], y_val=tweet_val[1],
    X_test=tweet_test[0], y_test=tweet_test[1],
    model_type="svm"
)

print("\n📊 TweetEval Results:")
print(
    f"   LR : Acc={tweet_lr['summary']['acc']['mean']:.4f} ± {tweet_lr['summary']['acc']['std']:.4f} | "
    f"F1={tweet_lr['summary']['f1']['mean']:.4f} ± {tweet_lr['summary']['f1']['std']:.4f} | "
    f"Train={tweet_lr['summary']['train_time']['mean']:.2f}s | "
    f"Inf={tweet_lr['summary']['inf_time']['mean']*1000:.2f}ms/sample"
)
print(
    f"   SVM: Acc={tweet_svm['summary']['acc']['mean']:.4f} ± {tweet_svm['summary']['acc']['std']:.4f} | "
    f"F1={tweet_svm['summary']['f1']['mean']:.4f} ± {tweet_svm['summary']['f1']['std']:.4f} | "
    f"Train={tweet_svm['summary']['train_time']['mean']:.2f}s | "
    f"Inf={tweet_svm['summary']['inf_time']['mean']*1000:.2f}ms/sample"
)

In [ ]:
# 4: FastText (Tier 2 Neural Model)

import fasttext
import tempfile
import os
import time
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Reproducibility helper
def set_seed(seed=42):
    np.random.seed(seed)

def _fmt_fasttext(texts, labels):
    """Format data for FastText: __label__<int> <text>"""
    cleaned = [str(t).replace('\n', ' ').replace('\r', ' ').strip() for t in texts]
    return [f"__label__{str(l)} {t}" for t, l in zip(cleaned, labels)]

def _extract_text(formatted_texts):
    """Extract raw text from __label__ format for prediction"""
    return [t.split(' ', 1)[1].strip() if ' ' in t else t for t in formatted_texts]

def train_fasttext(X_train, y_train, X_val, y_val, X_test, y_test, seed=42):
    """Train FastText with hyperparameter tuning - matches Cell 4 implementation"""
    set_seed(seed)
    
    # Format data
    train_fmt = _fmt_fasttext(X_train, y_train)
    val_fmt = _fmt_fasttext(X_val, y_val)
    val_texts = _extract_text(val_fmt)
    
    # Hyperparameter grid (expanded to match original)
    lr_opts = [0.1, 0.5, 1.0]
    epoch_opts = [5, 10, 15]
    
    best_val_f1 = -1.0
    best_params = None
    best_model = None
    best_train_time = 0
    
    print("🔍 Tuning FastText hyperparameters...")
    
    for lr in lr_opts:
        for epoch in epoch_opts:
            with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt', encoding='utf-8') as f:
                f.write('\n'.join(train_fmt))
                tmp_path = f.name
            
            t0 = time.time()
            model = fasttext.train_supervised(
                input=tmp_path, 
                lr=lr, 
                epoch=epoch, 
                wordNgrams=2, 
                verbose=0, 
                seed=seed, 
                loss='softmax', 
                dim=100,
                thread=1  # Ensure reproducibility
            )
            train_time = time.time() - t0
            os.remove(tmp_path)
            
            # Validate - batch prediction (efficient)
            preds_raw, _ = model.predict(val_texts, k=1)
            
            # Robust label extraction
            preds = []
            for p in preds_raw:
                label_str = p[0]
                if '__label__' in label_str:
                    preds.append(int(label_str.replace('__label__', '').strip()))
                else:
                    preds.append(int(label_str))
            
            val_f1 = f1_score(y_val, preds, average='macro')
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_params = (lr, epoch)
                best_model = model
                best_train_time = train_time
    
    print(f"✅ Best FastText: lr={best_params[0]}, epoch={best_params[1]} | Val F1: {best_val_f1:.4f}")
    
    # Test evaluation (no retraining - use best_model directly)
    test_fmt = _fmt_fasttext(X_test, y_test)
    test_texts = _extract_text(test_fmt)
    
    t0 = time.time()
    preds_raw, _ = best_model.predict(test_texts, k=1)
    test_preds = [int(p[0].replace('__label__', '').strip()) for p in preds_raw]
    inf_time = (time.time() - t0) / len(X_test)
    
    return {
        'best_lr': best_params[0],
        'best_epoch': best_params[1],
        'test_acc': accuracy_score(y_test, test_preds),
        'test_f1': f1_score(y_test, test_preds, average='macro'),
        'train_time': best_train_time,
        'inf_time': inf_time,
        'model': best_model  # Return model if needed for later use
    }

# -------------------------
# Run FastText on 20NG (Single seed - matches original Cell 4)
# -------------------------
print("🚀 Training FastText on 20 Newsgroups...")
news_ft = train_fasttext(
    X_train=news_train[0], y_train=news_train[1],
    X_val=news_val[0], y_val=news_val[1],
    X_test=news_test[0], y_test=news_test[1],
    seed=42
)
print(f"📊 20NG FastText: Acc={news_ft['test_acc']:.4f} | F1={news_ft['test_f1']:.4f} | Train={news_ft['train_time']:.2f}s | Inf={news_ft['inf_time']*1000:.2f}ms/sample")

# -------------------------
# Run FastText on TweetEval (Single seed - matches original Cell 4)
# -------------------------
print("\n🚀 Training FastText on TweetEval...")
tweet_ft = train_fasttext(
    X_train=tweet_train[0], y_train=tweet_train[1],
    X_val=tweet_val[0], y_val=tweet_val[1],
    X_test=tweet_test[0], y_test=tweet_test[1],
    seed=42
)
print(f"📊 TweetEval FastText: Acc={tweet_ft['test_acc']:.4f} | F1={tweet_ft['test_f1']:.4f} | Train={tweet_ft['train_time']:.2f}s | Inf={tweet_ft['inf_time']*1000:.2f}ms/sample")

In [ ]:
# 5: Tier 3 Transformer Models (DistilBERT & RoBERTa)

In [ ]:
import os, time, torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# Prevent CPU thrashing
torch.set_num_threads(4)

def train_transformer_tuned(model_name, X_train, y_train, X_val, y_val, X_test, y_test,
                            num_labels, id2label, seed=42):
    """Transformer training with LR tuning, early stopping, and per-class F1 extraction."""
    torch.manual_seed(seed); np.random.seed(seed)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)
        
    train_ds = Dataset.from_dict({"text": X_train, "label": y_train}).map(tokenize, batched=True)
    val_ds   = Dataset.from_dict({"text": X_val, "label": y_val}).map(tokenize, batched=True)
    test_ds  = Dataset.from_dict({"text": X_test, "label": y_test}).map(tokenize, batched=True)
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"acc": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average="macro")}
        
    label2id = {v: k for k, v in id2label.items()}
    
    # 🔍 HP TUNING: Grid search over LRs (Rubric requirement)
    best_lr, best_val_f1 = 2e-5, -1
    print(f"   🔍 Tuning LR for {model_name} (seed={seed})...")
    for lr in [1e-5, 2e-5, 5e-5]:
        args = TrainingArguments(
            output_dir="./tmp_tune", learning_rate=lr, per_device_train_batch_size=4,
            num_train_epochs=5, eval_strategy="epoch", load_best_model_at_end=True,
            metric_for_best_model="eval_f1", save_strategy="epoch", save_total_limit=1,
            disable_tqdm=True, report_to="none", fp16=False, seed=seed
        )
        trainer = Trainer(
            model=AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id2label, label2id=label2id),
            args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # 🟢 Early stopping
        )
        trainer.train()
        val_f1 = trainer.evaluate()["eval_f1"]
        if val_f1 > best_val_f1:
            best_val_f1, best_lr = val_f1, lr
            
    print(f"   ✅ Best LR: {best_lr} | Val F1: {best_val_f1:.4f}")
    
    # 🏁 Final training with best LR
    args = TrainingArguments(
        output_dir=f"./tmp_{model_name.split('/')[-1]}_final", learning_rate=best_lr,
        per_device_train_batch_size=4, num_train_epochs=5, eval_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_f1",
        save_strategy="epoch", save_total_limit=1, disable_tqdm=True, report_to="none",
        fp16=False, seed=seed
    )
    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id2label, label2id=label2id),
        args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    
    t0 = time.time(); trainer.train(); train_time = time.time() - t0
    print("   📊 Evaluating on test set...")
    t0 = time.time(); test_res = trainer.evaluate(test_ds); inf_time = (time.time() - t0) / len(X_test)
    
    # 📊 Per-class F1 extraction (Rubric requirement)
    logits = trainer.predict(test_ds).predictions
    preds = np.argmax(logits, axis=-1)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    cls_f1 = {id2label.get(str(k), str(k)): v['f1-score'] for k, v in report.items() if str(k).isdigit() and int(k) < num_labels}
    
    return {
        "test_acc": test_res["eval_acc"],
        "test_f1": test_res["eval_f1"],
        "train_time": train_time,
        "inf_time": inf_time,
        "best_class": max(cls_f1, key=cls_f1.get),
        "worst_class": min(cls_f1, key=cls_f1.get)
    }

# ================= RUN ON 20 NEWGROUPS =================
print("\n🚀 DistilBERT on 20 Newsgroups (CPU-Optimized, Multi-Seed)...")
news_dist = run_with_seeds(
    train_transformer_tuned, seeds=[42, 123, 7],
    model_name="distilbert-base-uncased",
    X_train=news_train[0], y_train=news_train[1],
    X_val=news_val[0], y_val=news_val[1],
    X_test=news_test[0], y_test=news_test[1],
    num_labels=20, id2label={i: str(i) for i in range(20)}
)
print(f"📊 20NG DistilBERT: Acc={news_dist['summary']['acc']['mean']:.4f} ± {news_dist['summary']['acc']['std']:.4f} | F1={news_dist['summary']['f1']['mean']:.4f} ± {news_dist['summary']['f1']['std']:.4f} | Train={news_dist['summary']['train_time']['mean']:.1f}s | Inf={news_dist['summary']['inf_time']['mean']*1000:.1f}ms/sample")

# ================= RUN ON TWEETEVAL =================
print("\n🚀 DistilBERT on TweetEval (CPU-Optimized, Multi-Seed)...")
tweet_dist = run_with_seeds(
    train_transformer_tuned, seeds=[42, 123, 7],
    model_name="distilbert-base-uncased",
    X_train=tweet_train[0], y_train=tweet_train[1],
    X_val=tweet_val[0], y_val=tweet_val[1],
    X_test=tweet_test[0], y_test=tweet_test[1],
    num_labels=2, id2label={0: "non_ironic", 1: "ironic"}
)
print(f"📊 TweetEval DistilBERT: Acc={tweet_dist['summary']['acc']['mean']:.4f} ± {tweet_dist['summary']['acc']['std']:.4f} | F1={tweet_dist['summary']['f1']['mean']:.4f} ± {tweet_dist['summary']['f1']['std']:.4f} | Train={tweet_dist['summary']['train_time']['mean']:.1f}s | Inf={tweet_dist['summary']['inf_time']['mean']*1000:.1f}ms/sample")

In [ ]:
import os, time, torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# Prevent CPU thrashing
torch.set_num_threads(4)

def train_transformer_tuned(model_name, X_train, y_train, X_val, y_val, X_test, y_test,
                            num_labels, id2label, seed=42):
    """Transformer training with LR tuning, early stopping, and per-class F1 extraction."""
    torch.manual_seed(seed); np.random.seed(seed)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)
        
    train_ds = Dataset.from_dict({"text": X_train, "label": y_train}).map(tokenize, batched=True)
    val_ds   = Dataset.from_dict({"text": X_val, "label": y_val}).map(tokenize, batched=True)
    test_ds  = Dataset.from_dict({"text": X_test, "label": y_test}).map(tokenize, batched=True)
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"acc": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average="macro")}
        
    label2id = {v: k for k, v in id2label.items()}
    
    # 🔍 HP TUNING: Grid search over LRs (Rubric requirement)
    best_lr, best_val_f1 = 2e-5, -1
    print(f"   🔍 Tuning LR for {model_name} (seed={seed})...")
    for lr in [1e-5, 2e-5, 5e-5]:
        args = TrainingArguments(
            output_dir="./tmp_tune", learning_rate=lr, per_device_train_batch_size=4,
            num_train_epochs=5, eval_strategy="epoch", load_best_model_at_end=True,
            metric_for_best_model="eval_f1", save_strategy="epoch", save_total_limit=1,
            disable_tqdm=True, report_to="none", fp16=False, seed=seed
        )
        trainer = Trainer(
            model=AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id2label, label2id=label2id),
            args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # 🟢 Early stopping
        )
        trainer.train()
        val_f1 = trainer.evaluate()["eval_f1"]
        if val_f1 > best_val_f1:
            best_val_f1, best_lr = val_f1, lr
            
    print(f"   ✅ Best LR: {best_lr} | Val F1: {best_val_f1:.4f}")
    
    # 🏁 Final training with best LR
    args = TrainingArguments(
        output_dir=f"./tmp_{model_name.split('/')[-1]}_final", learning_rate=best_lr,
        per_device_train_batch_size=4, num_train_epochs=5, eval_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_f1",
        save_strategy="epoch", save_total_limit=1, disable_tqdm=True, report_to="none",
        fp16=False, seed=seed
    )
    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id2label, label2id=label2id),
        args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    
    t0 = time.time(); trainer.train(); train_time = time.time() - t0
    print("   📊 Evaluating on test set...")
    t0 = time.time(); test_res = trainer.evaluate(test_ds); inf_time = (time.time() - t0) / len(X_test)
    
    # 📊 Per-class F1 extraction (Rubric requirement)
    logits = trainer.predict(test_ds).predictions
    preds = np.argmax(logits, axis=-1)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    cls_f1 = {id2label.get(str(k), str(k)): v['f1-score'] for k, v in report.items() if str(k).isdigit() and int(k) < num_labels}
    
    return {
        "test_acc": test_res["eval_acc"],
        "test_f1": test_res["eval_f1"],
        "train_time": train_time,
        "inf_time": inf_time,
        "best_class": max(cls_f1, key=cls_f1.get),
        "worst_class": min(cls_f1, key=cls_f1.get)
    }

# ================= RUN ON 20 NEWGROUPS =================
print("\n🚀 DistilBERT on 20 Newsgroups (CPU-Optimized, Multi-Seed)...")
news_dist = run_with_seeds(
    train_transformer_tuned, seeds=[42, 123, 7],
    model_name="distilbert-base-uncased",
    X_train=news_train[0], y_train=news_train[1],
    X_val=news_val[0], y_val=news_val[1],
    X_test=news_test[0], y_test=news_test[1],
    num_labels=20, id2label={i: str(i) for i in range(20)}
)
print(f"📊 20NG DistilBERT: Acc={news_dist['summary']['acc']['mean']:.4f} ± {news_dist['summary']['acc']['std']:.4f} | F1={news_dist['summary']['f1']['mean']:.4f} ± {news_dist['summary']['f1']['std']:.4f} | Train={news_dist['summary']['train_time']['mean']:.1f}s | Inf={news_dist['summary']['inf_time']['mean']*1000:.1f}ms/sample")

# ================= RUN ON TWEETEVAL =================
print("\n🚀 DistilBERT on TweetEval (CPU-Optimized, Multi-Seed)...")
tweet_dist = run_with_seeds(
    train_transformer_tuned, seeds=[42, 123, 7],
    model_name="distilbert-base-uncased",
    X_train=tweet_train[0], y_train=tweet_train[1],
    X_val=tweet_val[0], y_val=tweet_val[1],
    X_test=tweet_test[0], y_test=tweet_test[1],
    num_labels=2, id2label={0: "non_ironic", 1: "ironic"}
)
print(f"📊 TweetEval DistilBERT: Acc={tweet_dist['summary']['acc']['mean']:.4f} ± {tweet_dist['summary']['acc']['std']:.4f} | F1={tweet_dist['summary']['f1']['mean']:.4f} ± {tweet_dist['summary']['f1']['std']:.4f} | Train={tweet_dist['summary']['train_time']['mean']:.1f}s | Inf={tweet_dist['summary']['inf_time']['mean']*1000:.1f}ms/sample")